In [7]:
import pandas as pd
import numpy as np
from pyarrow import dataset as ds
import pickle

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.ensemble import HistGradientBoostingClassifier

### load data

In [8]:
dataset = ds.dataset("../data/renovation-data-all.parquet", format="parquet")
table = dataset.to_table(columns=None, use_threads=True)
df_preprocessed = table.to_pandas()
print(f"shape: {df_preprocessed.shape}")
df_preprocessed.head()

shape: (39736, 83)


,entity_id,epc_before,ontology.building.building_general.construction_year,ontology.building.building_general.floor_area,ontology.building.pv.available_roof_area_for_solar_panel_in_m2,ontology.building.pv.has_pv,steps,has_insulate_exterior_inclined_roof,has_insulate_exterior_flat_roof,has_insulate_exterior_wall,...,ontology.building.systems.space_heating_emission_systems.CONVECTOR,ontology.building.systems.space_heating_emission_systems.FLOOR,ontology.building.systems.space_heating_emission_systems.AIR,ontology.building.systems.space_heating_emission_systems.LOCAL_STOVE,ontology.building.envelope.insulation_level_roof_extension_MODERATELY_INSULATED,ontology.building.envelope.insulation_level_roof_main_MODERATELY_INSULATED,ontology.building.systems.space_heating_generation_on_electricity.CENTRAL_HEAT_PUMP,ontology.building.systems.space_heating_generation_on_electricity.LOCAL_RESISTANCE_HEATERS,ontology.building.envelope.basement_occurance_PARTIAL,ontology.building.envelope.floor_boundary_types.BASEMENT_OR_CRAWLSPACE
0,0,394,1965.0,203.0,8.0,0,"[insulate_floors_above_air_from_below, insulat...",1,1,1,...,1,0,0,0,0,0,0,0,0,1
1,1,413,1969.0,226.0,16.0,0,"[insulate_floors_above_air_from_below, insulat...",1,1,1,...,1,0,1,0,1,0,0,0,0,1
2,2,469,1967.0,195.0,71.0,0,"[insulate_exterior_inclined_roof, replace_sing...",1,1,1,...,0,0,0,0,1,0,0,0,0,1
3,3,592,1969.0,220.0,63.0,0,"[insulate_floors_above_air_from_below, replace...",0,1,0,...,1,1,0,0,0,0,0,0,0,1
4,5,511,1967.0,173.0,25.0,0,"[replace_epump_with_dhw, insulate_floors_above...",1,1,1,...,0,0,0,1,1,1,0,0,0,1


### train model

In [9]:
steps_encoding = {
    'insulate_exterior_inclined_roof': 1,
    'insulate_exterior_flat_roof': 2,
    'insulate_exterior_wall': 3,
    'insulate_cavity_wall': 4,
    'insulate_floors_above_air_from_below': 5,
    'replace_single_and_old_double_for_hr_glass_with_new_frames': 6,
    'replace_non_condensing_gas_boiler_with_condensing_gas_boiler': 7,
    'replace_epump_with_dhw': 8,
    'install_pv_panels': 9,  
}

def get_columns_to_drop(df, component):
    columns_to_drop = ["entity_id"]

    if component == "relevant_steps":
        columns_to_drop.extend([col for col in df.columns if col.startswith("has_")])
        columns_to_drop.extend(["step_count", "steps", "steps_gt"])
    elif component == "first_step":
        columns_to_drop.extend(["first_step", "steps", "steps_gt"]) # remove steps(gt) since list, but same information is in 'has_' columns

    return columns_to_drop

In [10]:
models_HGB = {}
reports = {}
oof_predictions = {}

HGB_params = {
    'random_state': 42,
    'learning_rate': 0.02,
    'max_iter': 800,
    'max_leaf_nodes':127,
    'min_samples_leaf': 5,
    'l2_regularization': 1.0,
    'class_weight': 'balanced',
}

# define features
columns_to_drop = get_columns_to_drop(df_preprocessed, component="relevant_steps")
X = df_preprocessed.drop(columns=columns_to_drop)
print(f"{X.shape[1]} features used for training: {X.columns.tolist()}")

for action in steps_encoding.keys():
    # define target
    target = f"has_{action}"
    y = df_preprocessed[target]

    ###################################
    ##### out-of-fold predictions #####
    ###################################

    # using StratifiedKFold to maintain class distribution in each fold
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    oof_proba = np.zeros(len(y))

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
        X_tr, X_va = X.iloc[train_idx], X.iloc[valid_idx]
        y_tr, y_va = y[train_idx], y[valid_idx]

        HGB_model = HistGradientBoostingClassifier(**HGB_params)

        HGB_model.fit(X_tr, y_tr)

        # store OOF prediction p(class=1)
        oof_proba[valid_idx] = HGB_model.predict_proba(X_va)[:, 1]

    # calculate F1 scores
    y_pred = (oof_proba >= 0.5).astype(int)

    f1_pos = f1_score(y, y_pred, pos_label=1)
    f1_neg = f1_score(y, y_pred, pos_label=0)

    # save oof predictions
    oof_predictions[target] = oof_proba
    reports[target] = classification_report(y, y_pred, output_dict=True)

    ################################
    ##### training final model #####
    ################################
    HGB_model_final = HistGradientBoostingClassifier(**HGB_params)
    HGB_model_final.fit(X, y)
    models_HGB[target] = HGB_model_final

# save models to file
filename = '../models/relevant_steps_HGB.pkl'
with open(filename, 'wb') as file:
    pickle.dump({
        "models_HGB": models_HGB,
        "oof_predictions": oof_predictions
    }, file)

print("Saved:", filename)

70 features used for training: ['epc_before', 'ontology.building.building_general.construction_year', 'ontology.building.building_general.floor_area', 'ontology.building.pv.available_roof_area_for_solar_panel_in_m2', 'ontology.building.pv.has_pv', 'ontology.building.energy_carriers.cooking.GAS_DISTRICT', 'ontology.building.energy_carriers.cooking.ELECTRICITY', 'ontology.building.energy_carriers.space_cooling.ELECTRICITY', 'ontology.building.systems.hot_water_system_types_on_electricity.ELECTRICAL_RESISTANCE', 'ontology.building.systems.hot_water_system_types_on_electricity.HEAT_PUMP', 'ontology.building.energy_carriers.space_heating.GAS_DISTRICT', 'ontology.building.energy_carriers.space_heating.COAL', 'ontology.building.energy_carriers.space_heating.OIL', 'ontology.building.energy_carriers.space_heating.ELECTRICITY', 'ontology.building.energy_carriers.space_heating.WOOD', 'ontology.building.energy_carriers.space_heating.PELLETS', 'ontology.building.envelope.floor_boundary_types.PLAIN_

### analyse results

In [34]:
results_df = pd.DataFrame(columns=["target", "F1 score (0)", "F1 score (1)", "Support (0)", "Support (1)"])
results_df["target"] = [key for key in reports.keys()]
results_df["F1 score (0)"] = [reports[key]["0"]["f1-score"] for key in reports.keys()]
results_df["F1 score (1)"] = [reports[key]["1"]["f1-score"] for key in reports.keys()]
results_df["Support (0)"] = [int(reports[key]["0"]["support"]) for key in reports.keys()]
results_df["Support (1)"] = [int(reports[key]["1"]["support"]) for key in reports.keys()]

display(results_df.round(3))

weighted_average = {
    '0': sum(reports[f"has_{key}"]['0']['f1-score'] * reports[f"has_{key}"]['0']['support'] for key in steps_encoding) / 
         sum(reports[f"has_{key}"]['0']['support'] for key in steps_encoding),
    '1': sum(reports[f"has_{key}"]['1']['f1-score'] * reports[f"has_{key}"]['1']['support'] for key in steps_encoding) / 
         sum(reports[f"has_{key}"]['1']['support'] for key in steps_encoding)
}

print(f"Weighted average F1 score (0): {weighted_average['0']:.3f}")
print(f"Weighted average F1 score (1): {weighted_average['1']:.3f}")
print(f"Mean weighted average F1 score: {np.mean([reports[key]["weighted avg"]["f1-score"] for key in reports.keys()]):.3f}")

,target,F1 score (0),F1 score (1),Support (0),Support (1)
0,has_insulate_exterior_inclined_roof,0.853,0.905,14967,24769
1,has_insulate_exterior_flat_roof,0.893,0.902,18273,21463
2,has_insulate_exterior_wall,0.875,0.916,15192,24544
3,has_insulate_cavity_wall,0.930,0.788,31366,8370
4,has_insulate_floors_above_air_from_below,0.744,0.913,8714,31022
5,has_replace_single_and_old_double_for_hr_glass...,0.904,0.916,18785,20951
6,has_replace_non_condensing_gas_boiler_with_con...,0.873,0.534,34310,5426
7,has_replace_epump_with_dhw,0.982,0.897,34058,5678
8,has_install_pv_panels,0.946,0.994,4071,35665


Weighted average F1 score (0): 0.903
Weighted average F1 score (1): 0.909
Mean weighted average F1 score: 0.906
